# 타겟보드 바이너리 컴파일 관련 파일 초기화 (make clean)

In [ ]:
%%bash
cd simpleserial_main/
make PLATFORM='CWHUSKY' CRYPTO_TARGET='NONE' SS_VER='SS_VER_2_1' clean
make PLATFORM='CWHUSKY' CRYPTO_TARGET='NONE' SS_VER='SS_VER_1_1' clean
make PLATFORM='CW308_STM32F3' CRYPTO_TARGET='NONE' SS_VER='SS_VER_2_1' clean
make PLATFORM='CW308_STM32F3' CRYPTO_TARGET='NONE' SS_VER='SS_VER_1_1' clean

# 기본 설정 파라미터

In [ ]:
SCOPETYPE = 'OPENADC'
CRYPTO_TARGET='NONE'
SS_VER='SS_VER_2_1'

# 주피터 노트북 그래프 plot 기능 활성화

In [ ]:
%matplotlib inline

# 타겟보드 바이너리 클린 & 빌드

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"

cd simpleserial_main/

make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 clean

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"

cd simpleserial_main/

make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3

# 칩위스퍼러 기본 셋업 (칩위스퍼러 원본 파일 유지) & 타겟보드 프로그램

In [ ]:
%run '../base/Setup_Generic.ipynb'

In [ ]:
cw.program_target(scope, prog, f'simpleserial_main/simpleserial-base-{PLATFORM}.hex')

# 타겟보드 바이너리 컴파일 관련 파일 정리 (make clean)

In [ ]:
%%bash
cd simpleserial_main/
make PLATFORM='CWHUSKY' CRYPTO_TARGET='NONE' SS_VER='SS_VER_2_1' clean
make PLATFORM='CWHUSKY' CRYPTO_TARGET='NONE' SS_VER='SS_VER_1_1' clean
make PLATFORM='CW308_STM32F3' CRYPTO_TARGET='NONE' SS_VER='SS_VER_2_1' clean
make PLATFORM='CW308_STM32F3' CRYPTO_TARGET='NONE' SS_VER='SS_VER_1_1' clean

# 패키지 및 개인 함수

In [ ]:
from tqdm.notebook import trange
from tqdm import tqdm
import numpy as np
import scipy as sp
import time
import math
import random
import pandas as pd
from matplotlib import pyplot as plt
import h5py
from pathlib import Path
import logging
import sys

- 칩위스퍼러 UFO보드(cw308) 타겟보드 리셋

In [ ]:
def cwUFO_reboot_flush():
    reset_target(scope)
    target.reset_comms()
    target.flush()

- CW310 데이터 쓰기 & 읽기 검증

In [ ]:
def cw310_write(target, addr, data):
    """
    CW310 타겟에 128바이트 데이터를 32바이트 청크로 나누어 쓰고,
    해당 주소에서 다시 데이터를 읽어오는 함수.

    Args:
        target: ChipWhisperer 타겟 객체 (cw.target 등)
        addr: 쓰기/읽기를 수행할 레지스터 주소 (int)
        data: 쓸 데이터 (list 또는 bytearray, 반드시 128바이트여야 함)

    Returns:
        list: 타겟에서 읽어온 데이터 (ret)
    """
    
    # 상수 정의
    MAX_CW310_BYTE_SIZE = 128
    MAX_USB_BYTE_SIZE = 32

    # 1. 입력 데이터 길이 검증
    if len(data) != MAX_CW310_BYTE_SIZE:
        raise ValueError(f"Data length must be exactly {MAX_CW310_BYTE_SIZE} bytes. (Current: {len(data)})")

    # 2. 주소 계산 (Byte addressing alignment)
    addr_s = addr << target.bytecount_size

    # 3. 32바이트 단위로 분할하여 쓰기 (Write)
    num_chunks = MAX_CW310_BYTE_SIZE // MAX_USB_BYTE_SIZE
    
    for i in range(num_chunks):
        offset = i * MAX_USB_BYTE_SIZE
        chunk = data[offset : offset + MAX_USB_BYTE_SIZE]
        
        # 하위 레벨 USB 명령어를 통해 메모리 쓰기
        target._naeusb.cmdWriteMem(addr_s + offset, chunk)

    # 4. 데이터 읽기 (Read)
    ret = target.fpga_read(addr, MAX_CW310_BYTE_SIZE)

    return ret



In [ ]:
def my_fsr_cmd(target, cmd, scmd, data, payload_only=False):
    target.flush()
    target.send_cmd(cmd=cmd, scmd=ord(scmd), data=data)

    response = target.read_cmd(timeout=500)

    if payload_only:
        return response[3:3 + response[2]]

    return response

In [ ]:
def my_setting_num_samples(target, scope, NumOfpresamples):  
    scope.arm()
    if my_fsr_cmd(target, 0x82, 'c', [], payload_only=True)[0] != 0x82:
        print('Crypt fail!')
        sys.exit('Crypt fail!')
    if scope.capture():
        print('Target timed out!')
        sys.exit('Target timed out!')

    scope.adc.presamples = NumOfpresamples
    scope.adc.samples = (scope.adc.trig_count + (2*scope.adc.presamples)) // scope.adc.decimate  

    return scope.adc.samples

In [ ]:
def my_get_trace(target, scope):  
    scope.arm()
    if my_fsr_cmd(target, 0x82, 'c', [], payload_only=True)[0] != 0x82:
        raise ValueError('Crypt fail!')
    if scope.capture():
        raise TimeoutError('Target timed out!')

    ret_o = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

    ret_tr = np.array(scope.get_last_trace())

    return ret_o, ret_tr

In [ ]:
print('Done! \n')